# Colombo Stock Exchange (CSE) Stock Market Analysis

A comprehensive Python notebook for analyzing stocks listed on the Colombo Stock Exchange.

## Features
- **Data Fetching**: Retrieve stock data from multiple sources
- **Technical Analysis**: Calculate popular indicators (MA, RSI, MACD, Bollinger Bands)
- **Visualization**: Interactive charts for price movements and indicators
- **Portfolio Analysis**: Track and analyze your portfolio performance

---

## 1. Setup and Dependencies

First, let's install and import the required libraries.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn yfinance pandas-ta plotly requests beautifulsoup4

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Try to import optional libraries
try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except ImportError:
    YFINANCE_AVAILABLE = False
    print("yfinance not installed. Some features may be limited.")

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("plotly not installed. Using matplotlib for visualization.")

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully!")

## 2. CSE Stock Data Configuration

List of popular CSE stocks with their Yahoo Finance ticker symbols.
Note: CSE stocks on Yahoo Finance typically have `.CM` suffix (e.g., `JKH.CM` for John Keells Holdings)

In [ ]:
# CSE Stock Tickers (Yahoo Finance format with .CM suffix)
CSE_STOCKS = {
    # Banking & Finance
    'COMB.CM': 'Commercial Bank of Ceylon',
    'SAMP.CM': 'Sampath Bank',
    'HNB.CM': 'Hatton National Bank',
    'NDB.CM': 'National Development Bank',
    'DFCC.CM': 'DFCC Bank',
    
    # Blue Chips / Conglomerates
    'JKH.CM': 'John Keells Holdings',
    'HPFL.CM': 'Hayleys PLC',
    'AITK.CM': 'Aitken Spence PLC',
    'CARG.CM': 'Cargills Ceylon',
    
    # Telecommunications
    'DIAL.CM': 'Dialog Axiata',
    'SLTL.CM': 'Sri Lanka Telecom',
    
    # Plantations
    'CALT.CM': 'Ceylon Tobacco Company',
    
    # Manufacturing
    'TILE.CM': 'Lanka Tiles',
    'RGEM.CM': 'Royal Ceramics',
    
    # Insurance
    'CINS.CM': 'Ceylinco Insurance',
    'UNIL.CM': 'Union Assurance',
}

print(f"Configured {len(CSE_STOCKS)} CSE stocks for analysis")
print("\nAvailable stocks:")
for ticker, name in CSE_STOCKS.items():
    print(f"  {ticker}: {name}")

## 3. Data Fetching Functions

Functions to fetch stock data from Yahoo Finance and other sources.

In [ ]:
def fetch_stock_data(ticker, start_date=None, end_date=None, period='1y'):
    """
    Fetch historical stock data from Yahoo Finance.
    
    Parameters:
    -----------
    ticker : str
        Stock ticker symbol (e.g., 'JKH.CM')
    start_date : str, optional
        Start date in 'YYYY-MM-DD' format
    end_date : str, optional
        End date in 'YYYY-MM-DD' format
    period : str, optional
        Time period ('1d', '5d', '1mo', '3mo', '6mo', '1y', '2y', '5y', '10y', 'ytd', 'max')
        Used only if start_date is not provided
    
    Returns:
    --------
    pandas.DataFrame
        Historical stock data with OHLCV columns
    """
    if not YFINANCE_AVAILABLE:
        print("yfinance is not installed. Please install it using: pip install yfinance")
        return pd.DataFrame()
    
    try:
        stock = yf.Ticker(ticker)
        
        if start_date:
            data = stock.history(start=start_date, end=end_date)
        else:
            data = stock.history(period=period)
        
        if data.empty:
            print(f"No data found for {ticker}. The ticker might be invalid or delisted.")
            return pd.DataFrame()
        
        # Clean column names
        data.columns = [col.lower().replace(' ', '_') for col in data.columns]
        
        print(f"Fetched {len(data)} rows for {ticker}")
        return data
        
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()


def fetch_multiple_stocks(tickers, start_date=None, end_date=None, period='1y'):
    """
    Fetch data for multiple stocks.
    
    Returns:
    --------
    dict
        Dictionary with ticker as key and DataFrame as value
    """
    data = {}
    for ticker in tickers:
        df = fetch_stock_data(ticker, start_date, end_date, period)
        if not df.empty:
            data[ticker] = df
    return data


def get_stock_info(ticker):
    """
    Get detailed information about a stock.
    """
    if not YFINANCE_AVAILABLE:
        return {}
    
    try:
        stock = yf.Ticker(ticker)
        return stock.info
    except Exception as e:
        print(f"Error getting info for {ticker}: {e}")
        return {}

### Example: Fetch Sample Data

In [ ]:
# Example: Fetch data for John Keells Holdings
sample_ticker = 'JKH.CM'
sample_data = fetch_stock_data(sample_ticker, period='1y')

if not sample_data.empty:
    print(f"\nSample data for {sample_ticker}:")
    print(sample_data.head())
    print(f"\nData shape: {sample_data.shape}")
    print(f"Date range: {sample_data.index.min()} to {sample_data.index.max()}")

## 4. Technical Analysis Indicators

Calculate popular technical indicators for stock analysis.

In [ ]:
class TechnicalAnalysis:
    """
    A class for calculating technical analysis indicators.
    """
    
    @staticmethod
    def simple_moving_average(data, column='close', window=20):
        """Calculate Simple Moving Average (SMA)"""
        return data[column].rolling(window=window).mean()
    
    @staticmethod
    def exponential_moving_average(data, column='close', span=20):
        """Calculate Exponential Moving Average (EMA)"""
        return data[column].ewm(span=span, adjust=False).mean()
    
    @staticmethod
    def relative_strength_index(data, column='close', window=14):
        """
        Calculate Relative Strength Index (RSI).
        
        RSI = 100 - (100 / (1 + RS))
        RS = Average Gain / Average Loss
        """
        delta = data[column].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
        
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi
    
    @staticmethod
    def macd(data, column='close', fast=12, slow=26, signal=9):
        """
        Calculate Moving Average Convergence Divergence (MACD).
        
        Returns:
        --------
        tuple
            (MACD line, Signal line, MACD histogram)
        """
        ema_fast = data[column].ewm(span=fast, adjust=False).mean()
        ema_slow = data[column].ewm(span=slow, adjust=False).mean()
        
        macd_line = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        histogram = macd_line - signal_line
        
        return macd_line, signal_line, histogram
    
    @staticmethod
    def bollinger_bands(data, column='close', window=20, num_std=2):
        """
        Calculate Bollinger Bands.
        
        Returns:
        --------
        tuple
            (Upper band, Middle band (SMA), Lower band)
        """
        sma = data[column].rolling(window=window).mean()
        std = data[column].rolling(window=window).std()
        
        upper_band = sma + (std * num_std)
        lower_band = sma - (std * num_std)
        
        return upper_band, sma, lower_band
    
    @staticmethod
    def average_true_range(data, window=14):
        """
        Calculate Average True Range (ATR) - volatility indicator.
        """
        high = data['high']
        low = data['low']
        close = data['close']
        
        tr1 = high - low
        tr2 = abs(high - close.shift())
        tr3 = abs(low - close.shift())
        
        true_range = np.maximum.reduce([tr1.values, tr2.values, tr3.values])
        true_range = pd.Series(true_range, index=data.index)
        atr = true_range.rolling(window=window).mean()
        
        return atr
    
    @staticmethod
    def volume_weighted_average_price(data):
        """
        Calculate Volume Weighted Average Price (VWAP).
        """
        typical_price = (data['high'] + data['low'] + data['close']) / 3
        return (typical_price * data['volume']).cumsum() / data['volume'].cumsum()
    
    @staticmethod
    def stochastic_oscillator(data, k_window=14, d_window=3):
        """
        Calculate Stochastic Oscillator (%K and %D).
        """
        low_min = data['low'].rolling(window=k_window).min()
        high_max = data['high'].rolling(window=k_window).max()
        
        stoch_k = ((data['close'] - low_min) / (high_max - low_min)) * 100
        stoch_d = stoch_k.rolling(window=d_window).mean()
        
        return stoch_k, stoch_d


def add_all_indicators(data):
    """
    Add all technical indicators to the dataframe.
    """
    ta = TechnicalAnalysis()
    
    # Moving Averages
    data['sma_20'] = ta.simple_moving_average(data, window=20)
    data['sma_50'] = ta.simple_moving_average(data, window=50)
    data['sma_200'] = ta.simple_moving_average(data, window=200)
    data['ema_12'] = ta.exponential_moving_average(data, span=12)
    data['ema_26'] = ta.exponential_moving_average(data, span=26)
    
    # RSI
    data['rsi'] = ta.relative_strength_index(data)
    
    # MACD
    data['macd'], data['macd_signal'], data['macd_hist'] = ta.macd(data)
    
    # Bollinger Bands
    data['bb_upper'], data['bb_middle'], data['bb_lower'] = ta.bollinger_bands(data)
    
    # ATR
    data['atr'] = ta.average_true_range(data)
    
    # Stochastic
    data['stoch_k'], data['stoch_d'] = ta.stochastic_oscillator(data)
    
    return data


print("Technical Analysis class loaded successfully!")

### Example: Calculate Indicators

In [ ]:
# Add indicators to sample data
if not sample_data.empty:
    sample_data_with_indicators = add_all_indicators(sample_data.copy())
    print("Indicators added to sample data:")
    print(sample_data_with_indicators.tail())

## 5. Visualization Functions

Create interactive and static charts for stock analysis.

In [ ]:
def plot_candlestick(data, title="Stock Price", show_volume=True):
    """
    Create an interactive candlestick chart using Plotly.
    Falls back to matplotlib if Plotly is not available.
    """
    if PLOTLY_AVAILABLE:
        if show_volume:
            fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                               vertical_spacing=0.03,
                               row_heights=[0.7, 0.3])
        else:
            fig = go.Figure()
        
        # Candlestick chart
        candlestick = go.Candlestick(
            x=data.index,
            open=data['open'],
            high=data['high'],
            low=data['low'],
            close=data['close'],
            name='Price'
        )
        
        if show_volume:
            fig.add_trace(candlestick, row=1, col=1)
            
            # Volume bars
            colors = ['red' if row['open'] > row['close'] else 'green' 
                      for idx, row in data.iterrows()]
            fig.add_trace(go.Bar(x=data.index, y=data['volume'], 
                                 marker_color=colors, name='Volume'),
                         row=2, col=1)
        else:
            fig.add_trace(candlestick)
        
        fig.update_layout(
            title=title,
            yaxis_title='Price (LKR)',
            xaxis_rangeslider_visible=False,
            template='plotly_white',
            height=600
        )
        
        fig.show()
    else:
        # Matplotlib fallback
        fig, axes = plt.subplots(2 if show_volume else 1, 1, 
                                  figsize=(14, 8 if show_volume else 6),
                                  sharex=True)
        
        if show_volume:
            ax1, ax2 = axes
        else:
            ax1 = axes
        
        ax1.plot(data.index, data['close'], label='Close Price', color='blue')
        ax1.fill_between(data.index, data['low'], data['high'], alpha=0.3)
        ax1.set_title(title)
        ax1.set_ylabel('Price (LKR)')
        ax1.legend()
        
        if show_volume:
            ax2.bar(data.index, data['volume'], alpha=0.7)
            ax2.set_ylabel('Volume')
        
        plt.tight_layout()
        plt.show()


def plot_with_indicators(data, ticker="Stock"):
    """
    Plot stock price with technical indicators.
    """
    fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
    
    # Price with Moving Averages and Bollinger Bands
    ax1 = axes[0]
    ax1.plot(data.index, data['close'], label='Close', linewidth=1.5)
    ax1.plot(data.index, data['sma_20'], label='SMA 20', alpha=0.7)
    ax1.plot(data.index, data['sma_50'], label='SMA 50', alpha=0.7)
    if 'bb_upper' in data.columns:
        ax1.fill_between(data.index, data['bb_lower'], data['bb_upper'], 
                         alpha=0.2, color='gray', label='Bollinger Bands')
    ax1.set_title(f'{ticker} - Price with Moving Averages')
    ax1.set_ylabel('Price (LKR)')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Volume
    ax2 = axes[1]
    colors = ['red' if data['close'].iloc[i] < data['open'].iloc[i] else 'green' 
              for i in range(len(data))]
    ax2.bar(data.index, data['volume'], color=colors, alpha=0.7)
    ax2.set_title('Volume')
    ax2.set_ylabel('Volume')
    ax2.grid(True, alpha=0.3)
    
    # RSI
    ax3 = axes[2]
    ax3.plot(data.index, data['rsi'], label='RSI', color='purple')
    ax3.axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
    ax3.axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
    ax3.fill_between(data.index, 30, 70, alpha=0.1, color='gray')
    ax3.set_title('Relative Strength Index (RSI)')
    ax3.set_ylabel('RSI')
    ax3.set_ylim(0, 100)
    ax3.legend(loc='upper left')
    ax3.grid(True, alpha=0.3)
    
    # MACD
    ax4 = axes[3]
    ax4.plot(data.index, data['macd'], label='MACD', color='blue')
    ax4.plot(data.index, data['macd_signal'], label='Signal', color='orange')
    ax4.bar(data.index, data['macd_hist'], label='Histogram', 
            color=['green' if x >= 0 else 'red' for x in data['macd_hist']], alpha=0.5)
    ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax4.set_title('MACD')
    ax4.set_ylabel('MACD')
    ax4.legend(loc='upper left')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_comparison(data_dict, column='close'):
    """
    Compare multiple stocks on the same chart.
    Normalizes prices to percentage change from first day.
    """
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for ticker, data in data_dict.items():
        # Normalize to percentage change
        normalized = (data[column] / data[column].iloc[0] - 1) * 100
        ax.plot(data.index, normalized, label=ticker, linewidth=1.5)
    
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    ax.set_title('Stock Comparison (Normalized % Change)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Percentage Change (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


print("Visualization functions loaded successfully!")

### Example: Visualize Stock Data

In [ ]:
# Plot candlestick chart
if not sample_data.empty:
    plot_candlestick(sample_data, title=f"{sample_ticker} - Candlestick Chart")

In [ ]:
# Plot with indicators
if not sample_data.empty and 'rsi' in sample_data_with_indicators.columns:
    plot_with_indicators(sample_data_with_indicators, ticker=sample_ticker)

## 6. Portfolio Management

Tools for tracking and analyzing your investment portfolio.

In [ ]:
class Portfolio:
    """
    A class for managing and analyzing a stock portfolio.
    """
    
    def __init__(self):
        self.holdings = pd.DataFrame(columns=['ticker', 'shares', 'avg_cost', 'purchase_date'])
        self.transactions = []
    
    def add_holding(self, ticker, shares, avg_cost, purchase_date=None):
        """
        Add a stock holding to the portfolio.
        """
        # Add new row using loc for efficiency
        new_row = {
            'ticker': ticker,
            'shares': shares,
            'avg_cost': avg_cost,
            'purchase_date': purchase_date or datetime.now().strftime('%Y-%m-%d')
        }
        self.holdings.loc[len(self.holdings)] = new_row
        self.transactions.append({
            'type': 'buy',
            'ticker': ticker,
            'shares': shares,
            'price': avg_cost,
            'date': purchase_date
        })
        print(f"Added {shares} shares of {ticker} at LKR {avg_cost:.2f}")
    
    def remove_holding(self, ticker, shares=None):
        """
        Remove a stock holding from the portfolio.
        If shares is None, removes all shares.
        """
        mask = self.holdings['ticker'] == ticker
        if mask.any():
            if shares is None:
                self.holdings = self.holdings[~mask]
            else:
                idx = self.holdings[mask].index[0]
                current_shares = self.holdings.loc[idx, 'shares']
                if shares >= current_shares:
                    self.holdings = self.holdings[~mask]
                else:
                    self.holdings.loc[idx, 'shares'] -= shares
            print(f"Removed {shares or 'all'} shares of {ticker}")
        else:
            print(f"{ticker} not found in portfolio")
    
    def get_current_value(self):
        """
        Calculate current portfolio value by fetching latest prices.
        """
        if self.holdings.empty:
            print("Portfolio is empty")
            return pd.DataFrame()
        
        portfolio_df = self.holdings.copy()
        current_prices = []
        
        for ticker in portfolio_df['ticker']:
            data = fetch_stock_data(ticker, period='5d')
            if not data.empty:
                current_prices.append(data['close'].iloc[-1])
            else:
                current_prices.append(np.nan)
        
        portfolio_df['current_price'] = current_prices
        portfolio_df['cost_basis'] = portfolio_df['shares'] * portfolio_df['avg_cost']
        portfolio_df['current_value'] = portfolio_df['shares'] * portfolio_df['current_price']
        portfolio_df['profit_loss'] = portfolio_df['current_value'] - portfolio_df['cost_basis']
        portfolio_df['profit_loss_pct'] = (portfolio_df['profit_loss'] / portfolio_df['cost_basis']) * 100
        
        return portfolio_df
    
    def get_summary(self):
        """
        Get a summary of portfolio performance.
        """
        portfolio_df = self.get_current_value()
        
        if portfolio_df.empty:
            return
        
        total_cost = portfolio_df['cost_basis'].sum()
        total_value = portfolio_df['current_value'].sum()
        total_pl = portfolio_df['profit_loss'].sum()
        total_pl_pct = (total_pl / total_cost) * 100 if total_cost > 0 else 0
        
        print("="*50)
        print("PORTFOLIO SUMMARY")
        print("="*50)
        print(f"Total Cost Basis:   LKR {total_cost:,.2f}")
        print(f"Current Value:      LKR {total_value:,.2f}")
        print(f"Total Profit/Loss:  LKR {total_pl:,.2f} ({total_pl_pct:+.2f}%)")
        print("="*50)
        
        return portfolio_df
    
    def plot_allocation(self):
        """
        Plot portfolio allocation pie chart.
        """
        portfolio_df = self.get_current_value()
        
        if portfolio_df.empty:
            return
        
        fig, ax = plt.subplots(figsize=(10, 8))
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(portfolio_df)))
        
        ax.pie(portfolio_df['current_value'], 
               labels=portfolio_df['ticker'],
               autopct='%1.1f%%',
               colors=colors,
               explode=[0.02]*len(portfolio_df))
        
        ax.set_title('Portfolio Allocation')
        plt.tight_layout()
        plt.show()


print("Portfolio class loaded successfully!")

### Example: Create a Sample Portfolio

In [ ]:
# Create a sample portfolio
my_portfolio = Portfolio()

# Add some sample holdings (adjust these to your actual holdings)
# my_portfolio.add_holding('JKH.CM', shares=100, avg_cost=150.00, purchase_date='2024-01-15')
# my_portfolio.add_holding('COMB.CM', shares=200, avg_cost=80.00, purchase_date='2024-02-01')
# my_portfolio.add_holding('DIAL.CM', shares=500, avg_cost=10.50, purchase_date='2024-03-10')

# Uncomment above lines and run to see portfolio analysis
# portfolio_summary = my_portfolio.get_summary()
# my_portfolio.plot_allocation()

## 7. Trading Signals & Analysis

Generate trading signals based on technical indicators.

In [ ]:
def generate_signals(data):
    """
    Generate trading signals based on technical indicators.
    
    Returns DataFrame with signal columns.
    """
    signals = data.copy()
    
    # RSI Signals
    signals['rsi_signal'] = 'hold'
    signals.loc[signals['rsi'] < 30, 'rsi_signal'] = 'buy'
    signals.loc[signals['rsi'] > 70, 'rsi_signal'] = 'sell'
    
    # MACD Signals
    signals['macd_signal_type'] = 'hold'
    signals.loc[(signals['macd'] > signals['macd_signal']) & 
                (signals['macd'].shift(1) <= signals['macd_signal'].shift(1)), 
                'macd_signal_type'] = 'buy'
    signals.loc[(signals['macd'] < signals['macd_signal']) & 
                (signals['macd'].shift(1) >= signals['macd_signal'].shift(1)), 
                'macd_signal_type'] = 'sell'
    
    # Moving Average Crossover
    signals['ma_signal'] = 'hold'
    signals.loc[(signals['sma_20'] > signals['sma_50']) & 
                (signals['sma_20'].shift(1) <= signals['sma_50'].shift(1)), 
                'ma_signal'] = 'buy'
    signals.loc[(signals['sma_20'] < signals['sma_50']) & 
                (signals['sma_20'].shift(1) >= signals['sma_50'].shift(1)), 
                'ma_signal'] = 'sell'
    
    # Bollinger Bands Signal
    signals['bb_signal'] = 'hold'
    signals.loc[signals['close'] < signals['bb_lower'], 'bb_signal'] = 'buy'
    signals.loc[signals['close'] > signals['bb_upper'], 'bb_signal'] = 'sell'
    
    return signals


def get_signal_summary(data):
    """
    Get the latest trading signal summary.
    """
    signals = generate_signals(data)
    latest = signals.iloc[-1]
    
    print("="*50)
    print("TRADING SIGNAL SUMMARY (Latest)")
    print("="*50)
    print(f"Date: {signals.index[-1]}")
    print(f"Close Price: LKR {latest['close']:.2f}")
    print(f"\nIndicator Signals:")
    print(f"  RSI ({latest['rsi']:.1f}):     {latest['rsi_signal'].upper()}")
    print(f"  MACD:          {latest['macd_signal_type'].upper()}")
    print(f"  MA Crossover:  {latest['ma_signal'].upper()}")
    print(f"  Bollinger:     {latest['bb_signal'].upper()}")
    print("="*50)
    
    # Consensus
    signal_counts = {'buy': 0, 'sell': 0, 'hold': 0}
    for sig in ['rsi_signal', 'macd_signal_type', 'ma_signal', 'bb_signal']:
        signal_counts[latest[sig]] += 1
    
    consensus = max(signal_counts, key=signal_counts.get)
    print(f"\nConsensus: {consensus.upper()} ({signal_counts[consensus]}/4 indicators)")
    
    return signals


print("Signal generation functions loaded successfully!")

In [ ]:
# Generate signals for sample data
if not sample_data.empty and 'rsi' in sample_data_with_indicators.columns:
    signals_df = get_signal_summary(sample_data_with_indicators)

## 8. Market Overview & Screening

Tools for screening and comparing multiple stocks.

In [ ]:
def screen_stocks(tickers, period='3mo'):
    """
    Screen multiple stocks and rank them based on performance.
    """
    results = []
    
    for ticker in tickers:
        data = fetch_stock_data(ticker, period=period)
        
        if data.empty:
            continue
        
        data_with_ind = add_all_indicators(data.copy())
        latest = data_with_ind.iloc[-1]
        
        # Calculate metrics
        return_pct = ((data['close'].iloc[-1] / data['close'].iloc[0]) - 1) * 100
        avg_volume = data['volume'].mean()
        volatility = data['close'].pct_change().std() * np.sqrt(252) * 100
        
        results.append({
            'ticker': ticker,
            'name': CSE_STOCKS.get(ticker, ticker),
            'close': latest['close'],
            'return_pct': return_pct,
            'rsi': latest['rsi'],
            'avg_volume': avg_volume,
            'volatility': volatility
        })
    
    df = pd.DataFrame(results)
    
    if not df.empty:
        df = df.sort_values('return_pct', ascending=False)
    
    return df


def find_oversold_stocks(tickers, rsi_threshold=30):
    """
    Find stocks that are currently oversold (RSI below threshold).
    """
    screened = screen_stocks(tickers)
    oversold = screened[screened['rsi'] < rsi_threshold]
    return oversold


def find_overbought_stocks(tickers, rsi_threshold=70):
    """
    Find stocks that are currently overbought (RSI above threshold).
    """
    screened = screen_stocks(tickers)
    overbought = screened[screened['rsi'] > rsi_threshold]
    return overbought


print("Stock screening functions loaded successfully!")

### Example: Screen Top CSE Stocks

In [ ]:
# Screen a few stocks (uncomment to run)
# Note: This may take a while as it fetches data for each stock

# sample_tickers = ['JKH.CM', 'COMB.CM', 'DIAL.CM', 'HNB.CM']
# screening_results = screen_stocks(sample_tickers, period='3mo')
# print("\nStock Screening Results:")
# print(screening_results.to_string(index=False))

## 9. Manual Data Entry

For CSE stocks that may not be available on Yahoo Finance, you can manually enter data.

In [ ]:
def create_manual_data(dates, opens, highs, lows, closes, volumes):
    """
    Create a DataFrame from manually entered data.
    
    Parameters:
    -----------
    dates : list
        List of dates in 'YYYY-MM-DD' format
    opens, highs, lows, closes, volumes : list
        OHLCV data
    
    Returns:
    --------
    pandas.DataFrame
    """
    data = pd.DataFrame({
        'open': opens,
        'high': highs,
        'low': lows,
        'close': closes,
        'volume': volumes
    }, index=pd.to_datetime(dates))
    
    return data


def load_data_from_csv(file_path, date_column='Date'):
    """
    Load stock data from a CSV file.
    
    Expected columns: Date, Open, High, Low, Close, Volume
    """
    try:
        data = pd.read_csv(file_path, parse_dates=[date_column])
        data = data.set_index(date_column)
        data.columns = [col.lower().replace(' ', '_') for col in data.columns]
        return data
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return pd.DataFrame()


print("Manual data entry functions loaded successfully!")

### Example: Manual Data Entry

In [ ]:
# Example: Create manual data for a stock
# Uncomment and modify with your actual data

# manual_dates = ['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']
# manual_opens = [100.0, 102.0, 101.5, 103.0]
# manual_highs = [103.0, 104.0, 103.0, 105.0]
# manual_lows = [99.5, 101.0, 100.0, 102.0]
# manual_closes = [102.0, 101.5, 102.5, 104.0]
# manual_volumes = [10000, 12000, 8000, 15000]

# manual_stock_data = create_manual_data(
#     manual_dates, manual_opens, manual_highs, manual_lows, manual_closes, manual_volumes
# )
# print(manual_stock_data)

## 10. Useful Resources & Notes

### Data Sources for CSE Stocks

1. **Yahoo Finance** - Limited coverage of CSE stocks (use `.CM` suffix)
2. **CSE Official Website** - https://www.cse.lk/
3. **LankaBangla Securities** - Research reports
4. **CAL Research** - Company analysis

### Important Notes

- CSE trading hours: 9:30 AM - 2:30 PM (Sri Lanka Standard Time)
- Market is closed on weekends and Sri Lankan public holidays
- Price quotes are in Sri Lankan Rupees (LKR)

### Disclaimer

**This notebook is for educational and personal analysis purposes only. It does not constitute financial advice. Always do your own research and consult with a qualified financial advisor before making investment decisions.**

---

## Quick Start Guide

1. **Fetch stock data:**
   ```python
   data = fetch_stock_data('JKH.CM', period='1y')
   ```

2. **Add technical indicators:**
   ```python
   data_with_indicators = add_all_indicators(data.copy())
   ```

3. **Visualize:**
   ```python
   plot_candlestick(data, title="JKH Candlestick")
   plot_with_indicators(data_with_indicators, ticker="JKH.CM")
   ```

4. **Get trading signals:**
   ```python
   signals = get_signal_summary(data_with_indicators)
   ```

5. **Track portfolio:**
   ```python
   portfolio = Portfolio()
   portfolio.add_holding('JKH.CM', 100, 150.00)
   portfolio.get_summary()
   ```

In [ ]:
# Your custom analysis starts here!
# Use the functions defined above to analyze CSE stocks.

print("Ready for your stock market analysis!")
print("\nExample:")
print("  data = fetch_stock_data('JKH.CM', period='1y')")
print("  data = add_all_indicators(data)")
print("  plot_with_indicators(data, 'JKH')")